# Game-Theoretic Modeling of Strategic Data Poisoning
## Min–Max Optimization in Collaborative Neural Networks

---

### Conference-Level Experimental Pipeline
This notebook reproduces the complete set of experiments for the research paper. It is designed to run end-to-end on Google Colab (T4 GPU recommended).

### Notebook Structure
| Section | Description | Expected Time (T4 GPU) |
|---------|-------------|-------------------------|
| 0 | Setup & Imports | 1 min |
| 1 | Quick Validation (MNIST) | 3 mins |
| 2 | Defense Comparison (CIFAR-10) | 30 mins |
| 3 | Hard Benchmark (CIFAR-100) | 2 hours |
| 4 | Federated Learning | 15 mins |
| 5 | Ablation Studies (Figures) | 15 mins |
| 6 | Multi-dataset Table (Final) | 1 min |


---
## Section 0: Setup & Imports


In [ ]:
# ── COLAB SETUP ──────────────────────────────────────────────────────────────
import subprocess, sys
def install(pkg):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

required = ['torch', 'torchvision', 'numpy', 'matplotlib', 'pandas', 'scipy', 'tqdm']
for pkg in required:
    try: __import__(pkg)
    except ImportError: install(pkg)


In [ ]:
import sys, os
import torch
import pandas as pd
import matplotlib.pyplot as plt

# ── Anchor PROJECT_ROOT to the repo root (robust for Colab clones & subdirs) ──
def _find_repo_root(marker='experiments'):
    """Walk up from cwd until a directory containing `marker` is found."""
    current = os.path.abspath('.')
    for _ in range(6):  # max 6 levels up
        if os.path.isdir(os.path.join(current, marker)):
            return current
        parent = os.path.dirname(current)
        if parent == current:
            break
        current = parent
    return os.path.abspath('.')  # fallback: use cwd

PROJECT_ROOT = _find_repo_root()
os.chdir(PROJECT_ROOT)  # all relative paths (./data, ./results) now resolve correctly
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
print(f'Project root: {PROJECT_ROOT}')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

# Create output directories
os.makedirs('results/figures', exist_ok=True)
os.makedirs('results/tables', exist_ok=True)


---
## Section 1: Quick Validation (MNIST)
Runs the baseline, poisoning attack, and defense on MNIST (CNN) to ensure the pipeline works.


In [ ]:
from experiments.baseline import run_baseline
from experiments.poisoning import run_poisoned
from experiments.defender import run_minmax

print("\n--- MNIST QUICK RUN ---")
mnist_base = run_baseline(n_runs=3, epochs=5, batch_size=64)
mnist_pois = run_poisoned(n_runs=3, epochs=5, batch_size=64, src_class=1, tgt_class=7, poison_fraction=0.5)
mnist_def  = run_minmax(n_rounds=3, n_runs=3, epochs=5, batch_size=64, src_class=1, tgt_class=7, poison_fraction=0.5)


---
## Section 2: Defense Comparison (CIFAR-10)
Compares our Min-Max defender against Spectral Signatures and SEVER on CIFAR-10 using ResNet-18.
This establishes the main defense comparison table for the paper.


In [ ]:
from experiments.defense_comparison import run_defense_comparison

print("\n--- CIFAR-10 HEAD-TO-HEAD COMP ---")
# Runs Clean, Poisoned, Spectral, SEVER, and Min-Max
cifar10_df = run_defense_comparison(n_runs=3, dataset='cifar10', epochs=10, defense_epochs=10)


---
## Section 3: Hard Benchmark (CIFAR-100)
Trains WideResNet-28-10 on CIFAR-100. This is the hardest benchmark and demonstrates scalability.


In [ ]:
from experiments.cifar100_experiment import run_cifar100_full_pipeline

print("\n--- CIFAR-100 FULL PIPELINE ---")
# Note: 100 epochs is standard for WRN-28-10. This takes time!
cifar100_results = run_cifar100_full_pipeline(n_runs=3, epochs=100, batch_size=128)


---
## Section 4: Federated Learning (FedMedian vs FedAvg)
Simulates 5 clients (1 Byzantine). Compares FedAvg (vulnerable) to FedMedian (robust).


In [ ]:
from federated.fl_experiment import run_fl_full_comparison

print("\n--- FEDERATED LEARNING / BYZANTINE ---")
fl_df = run_fl_full_comparison(n_runs=3, fl_rounds=10, local_epochs=3, dataset='cifar10')


---
## Section 5: Ablation Studies
Generates Figures 4-6 for the paper (Learning Rate sensitivity, Epochs/Round, and Robustness across Poison Fractions).


In [ ]:
from experiments.ablations import ablation_lr, ablation_epochs, ablation_defended_vs_fraction

print("\n--- ABLATION STUDIES ---")
ablation_lr(n_runs=3)
ablation_epochs(n_runs=3)
ablation_defended_vs_fraction(n_runs=3)


---
## Section 6: Multi-Dataset Master Table
Compiles all runs across MNIST, CIFAR-10, and CIFAR-100 into the final "Table 1" for the paper.


In [ ]:
from experiments.multidataset_comparison import build_multidataset_table
import pandas as pd

print("\n--- FINAL MULTI-DATASET TABLE ---")

def _row(df, method):
    row = df[df["Method"] == method]
    if len(row) == 0:
        return {"mean": 0.0, "std": 0.0}
    return {"mean": float(row["Mean Acc (%)"].values[0]),
            "std":  float(row["Std (%)"].values[0])}

# ── Collect results per dataset; skip gracefully if a section was not run ──
all_res = {}
_missing = []

# MNIST (Section 1)
try:
    all_res['mnist'] = {
        'baseline': mnist_base,
        'poisoned': mnist_pois,
        'defended': {'mean': mnist_def['final_summary']['mean'],
                     'std':  mnist_def['final_summary']['std']},
    }
except NameError as e:
    print(f"[WARN] MNIST results missing ({e}). Section 1 was not run. Skipping.")
    _missing.append('MNIST')

# CIFAR-10 (Section 2)
try:
    all_res['cifar10'] = {
        'baseline': _row(cifar10_df, 'Clean Baseline'),
        'poisoned': _row(cifar10_df, 'Poisoned (No Defense)'),
        'defended': _row(cifar10_df, 'Ours (Min–Max)'),
        'spectral': _row(cifar10_df, 'Spectral Signatures'),
        'sever':    _row(cifar10_df, 'SEVER'),
    }
except NameError as e:
    print(f"[WARN] CIFAR-10 results missing ({e}). Section 2 was not run. Skipping.")
    _missing.append('CIFAR-10')

# CIFAR-100 (Section 3)
try:
    all_res['cifar100'] = {
        'baseline': cifar100_results['baseline'],
        'poisoned': cifar100_results['poisoned'],
    }
except NameError as e:
    print(f"[WARN] CIFAR-100 results missing ({e}). Section 3 was not run. Skipping.")
    _missing.append('CIFAR-100')

# Build table (falls back to placeholder rows for any datasets not run)
if all_res:
    multidataset_df = build_multidataset_table(all_res)
else:
    print('[WARN] No experiment results available. Printing placeholder table.')
    multidataset_df = build_multidataset_table(None)

if _missing:
    print(f"\n[NOTE] Datasets absent from table (sections not run): {', '.join(_missing)}")
print("\nDone! Open paper/paper_structure.md and fill in the blanks using tables in results/tables/.")
